# Chapter 16 — Mechanistic Interpretability

**Math for ML Engineers** · companion notebook

This notebook mirrors §16.1 (superposition, the toy model, Theorem 16.1) and §16.3.1–16.3.2 (induction circuits and activation patching, Definitions 16.4–16.5) — two of the field's canonical results, chosen because together they show both *why* individual neurons resist interpretation and *how* interpretability researchers establish causal claims about circuits anyway. **Key takeaway: superposition is a geometric consequence of sparsity, not an accident of training, and activation patching turns "this component matters" from a guess into a falsifiable, causal experiment.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(16)

## Superposition: packing more features than dimensions

A linear layer with $n$ input features and $m < n$ hidden dimensions can only give each feature its own **private, orthogonal axis** if $n \le m$. When $n > m$, geometry forbids full orthogonality — but Theorem 16.1 shows the network doesn't need exact orthogonality, only *near*-orthogonality, and how well it can achieve that depends on how often features co-occur:

$$
\hat{x} = \mathrm{ReLU}(W^\top W x + b), \qquad W \in \mathbb{R}^{m \times n}
$$

$$
\mathcal{L}(W, b) = \mathbb{E}_x\!\left[\sum_{i=1}^n I_i \,(\hat{x}_i - x_i)^2\right], \qquad I_i = 0.9^{\,i}
$$

where $x$ is drawn with each feature independently zeroed out with probability `sparsity` (and otherwise a random positive magnitude), and $I_i$ down-weights the importance of higher-indexed features. When inputs are **dense** (sparsity low), most features co-occur, interference is costly, and the optimal $W$ gives the top $m$ features clean private axes and drops the rest. When inputs are **sparse**, features rarely co-occur, so $W$ can pack many more directions into the same $m$-dimensional space at the cost of small, rare interference — this is *superposition*. The diagnostic used below, dimensions-per-feature

$$
D_i = \frac{\lVert W_i \rVert^2}{\sum_j (\hat{W}_i \cdot W_j)^2}
$$

equals $1$ for a feature with a fully private axis and drops toward $0$ as more features are squeezed into the same directions.

In [ ]:
N_FEATURES, N_HIDDEN, BATCH, STEPS, LR = 8, 3, 1024, 4000, 1e-2


def sample_features(rng, batch, sparsity, n=N_FEATURES):
    # Each feature is independently "present" with prob (1 - sparsity), with a
    # random positive magnitude when present -- this is the sparse-input model
    # behind the toy model of superposition (Ch. 16, Elhage et al. 2022).
    return (rng.random((batch, n)) > sparsity) * rng.random((batch, n))


def toy_loss_and_grads(x, W, b, importance):
    """xhat = ReLU(W^T W x + b), importance-weighted MSE, exact gradients (Ch. 16 toy model)."""
    batch = x.shape[0]
    h = x @ W.T
    y = h @ W + b
    resid = np.maximum(y, 0.0) - x
    loss = float(np.mean(np.sum(importance * resid ** 2, axis=1)))
    g = 2.0 * importance * resid * (y > 0) / batch
    return loss, h.T @ g + (g @ W.T).T @ x, g.sum(axis=0)

## Training the toy model and measuring feature packing

We train $W$ and $b$ with Adam directly against the exact gradients of that importance-weighted loss for a fixed number of steps, then compute $D_i$ (dimensions-per-feature) for every feature. Sweeping `sparsity` from $0.0$ (fully dense) to $0.99$ (rare features) and re-training from scratch at each level reproduces the **phase-change curve**: at low sparsity the model can only afford private axes for $N_{\text{hidden}} = 3$ features (mean $D_i \to 1.0$ among the "live" features it keeps), while at high sparsity all $8$ features get represented with $D_i \ll 1$ — superposition.

In [ ]:
def train_toy(sparsity, seed=0):
    rng = np.random.default_rng(seed)
    W = rng.normal(scale=0.1, size=(N_HIDDEN, N_FEATURES))
    b = np.zeros(N_FEATURES)
    importance = 0.9 ** np.arange(N_FEATURES)
    mW, vW = np.zeros_like(W), np.zeros_like(W)
    mb, vb = np.zeros_like(b), np.zeros_like(b)
    for t in range(1, STEPS + 1):
        x = sample_features(rng, BATCH, sparsity)
        loss, dW, db = toy_loss_and_grads(x, W, b, importance)
        for p, dp, m_, v_ in ((W, dW, mW, vW), (b, db, mb, vb)):
            m_ *= 0.9;   m_ += 0.1 * dp
            v_ *= 0.999; v_ += 0.001 * dp * dp
            p -= LR * (m_ / (1 - 0.9 ** t)) / (np.sqrt(v_ / (1 - 0.999 ** t)) + 1e-8)
    return W, loss


def dims_per_feature(W):
    """D_i = ||W_i||^2 / sum_j (W_i_hat . W_j)^2. D_i=1 means a private axis."""
    norms = np.linalg.norm(W, axis=0)
    what = W / np.where(norms > 1e-9, norms, 1.0)
    return norms ** 2 / np.maximum(((what.T @ W) ** 2).sum(axis=1), 1e-12)


sparsity_levels = [0.0, 0.5, 0.8, 0.9, 0.99]
weight_matrices, mean_dims, final_losses = [], [], []

for sparsity in sparsity_levels:
    W, loss = train_toy(sparsity)
    D = dims_per_feature(W)
    live = np.linalg.norm(W, axis=0) > 0.1
    weight_matrices.append(W)
    mean_dims.append(D[live].mean() if live.any() else 0.0)
    final_losses.append(loss)
    print(f"sparsity={sparsity:5.2f}  final_loss={loss:.5f}  "
          f"live_features={int(live.sum())}/{N_FEATURES}  mean D_i (live)={mean_dims[-1]:.3f}")

## Visualizing the phase change and the packing geometry

Two views of the same phenomenon. First, **mean dimensions-per-feature vs. sparsity** — this is the single most-reproduced plot in the mechanistic interpretability literature (Elhage et al., 2022): it shows a visible bend from "one feature per dimension" toward "many features sharing dimensions" as sparsity increases. Second, a **heatmap of $W^\top W$** at the highest sparsity level, which reveals the actual packing geometry: large diagonal entries (each feature has substantial self-overlap $\approx 1$) alongside small but structured off-diagonal interference, often arranged so that co-active feature pairs get pushed toward *negative* or *near-zero* overlap while rarely-co-active pairs can share a direction more cheaply.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(sparsity_levels, mean_dims, "o-", color="C0", linewidth=2, markersize=8)
axes[0].axhline(1.0, color="gray", linestyle="--", linewidth=1, label="1.0 = fully private axis")
axes[0].axhline(N_HIDDEN / N_FEATURES, color="gray", linestyle=":", linewidth=1,
                label=f"{N_HIDDEN}/{N_FEATURES} = all features, evenly packed")
axes[0].set_xlabel("input sparsity")
axes[0].set_ylabel("mean dimensions-per-feature (live features)")
axes[0].set_title("The superposition phase change")
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 1.15)

W_dense_overlap = weight_matrices[-1].T @ weight_matrices[-1]
im = axes[1].imshow(W_dense_overlap, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_title(f"$W^\\top W$ at sparsity={sparsity_levels[-1]}\n(diagonal=self-overlap, off-diag=interference)")
axes[1].set_xlabel("feature j")
axes[1].set_ylabel("feature i")
plt.colorbar(im, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

print(f"At sparsity={sparsity_levels[0]}: mean D_i (live) = {mean_dims[0]:.3f} -- "
      f"close to 1.0, only the top-importance features get clean private axes.")
print(f"At sparsity={sparsity_levels[-1]}: mean D_i (live) = {mean_dims[-1]:.3f} -- "
      f"well below 1.0, all {N_FEATURES} features are represented by squeezing "
      f"them into just {N_HIDDEN} dimensions via superposition.")

## Why this toy model explains real LLM neurons

This 8-feature, 3-dimension toy is not a curiosity — it is the leading explanation for one of the most persistent obstacles in real-world LLM interpretability: **polysemanticity**. Individual neurons inside production transformers routinely fire for clusters of totally unrelated concepts (a neuron that activates for both "French text" and "the number seven" is a normal finding, not a bug in your probing code). Superposition explains *why*: real activation spaces have vastly more genuine features than they have dimensions, and because those features are sparse in any given input, the network's loss-minimizing solution is exactly what Theorem 16.1 predicts — pack many near-orthogonal feature directions into a comparatively low-dimensional residual stream, at the cost of neurons that respond to non-obvious combinations of unrelated concepts along their axis.

This is precisely why **sparse autoencoders (SAEs)** became the dominant interpretability tool for LLMs starting around 2023-2024. An SAE (Eq. 16.5) trains an *overcomplete* basis — many more learned latent directions than the model's hidden width — with an L1 (later JumpReLU/TopK) sparsity penalty, explicitly trying to *invert* the superposition compression: find the directions the model's own weights are packing together, and unpack them one-per-latent so that each learned feature direction is monosemantic (fires for one human-interpretable concept) even though the underlying neurons it's built from are not. The one-sided soft-threshold in Eq. (16.6) — clamping only the "off" direction, unlike the symmetric two-sided Lasso operator of Eq. (4.16) — matters for exactly this reason: SAE activations are meant to represent "feature present, with some magnitude," not a signed coefficient, so thresholding must be one-sided to match that semantics.

In [ ]:
# Numeric check: does the *number of interpretable, near-monosemantic directions*
# actually exceed the number of hidden dimensions once sparsity is high, and does
# it collapse back to <= N_HIDDEN at low sparsity? This is the concrete, checkable
# claim underneath "SAEs try to unpack more features than there are neurons."

def count_recoverable_features(W, overlap_thresh=0.2):
    """A feature i is 'recoverable' (cleanly separable) if its best off-diagonal
    overlap with any other feature stays below overlap_thresh -- i.e. querying its
    direction wouldn't strongly activate a different feature too."""
    norms = np.linalg.norm(W, axis=0)
    live = norms > 0.1
    what = W / np.where(norms > 1e-9, norms, 1.0)
    G = np.abs(what.T @ what)
    np.fill_diagonal(G, 0.0)
    G = G[np.ix_(live, live)]
    max_offdiag = G.max(axis=0) if live.sum() > 1 else np.zeros(int(live.sum()))
    return int(np.sum(max_offdiag < overlap_thresh)), int(live.sum())


print(f"{'sparsity':>9} | {'live features':>13} | {'mean D_i':>8} | {'cleanly-separable':>18} | {'> N_HIDDEN packed?':>19}")
print("-" * 84)
for sparsity, W, D in zip(sparsity_levels, weight_matrices, mean_dims):
    recoverable, live = count_recoverable_features(W)
    packed_more_than_hidden = live > N_HIDDEN
    print(f"{sparsity:9.2f} | {live:13d} | {D:8.3f} | {recoverable:18d} | {str(packed_more_than_hidden):>19}")

n_dense_live = int((np.linalg.norm(weight_matrices[0], axis=0) > 0.1).sum())
n_sparse_live = int((np.linalg.norm(weight_matrices[-1], axis=0) > 0.1).sum())
print(f"\nAt sparsity={sparsity_levels[0]}: {n_dense_live} live features fit in {N_HIDDEN} dims "
      f"(no superposition needed -- an SAE here would find nothing to unpack).")
print(f"At sparsity={sparsity_levels[-1]}: {n_sparse_live} live features share {N_HIDDEN} dims "
      f"({n_sparse_live - N_HIDDEN} more features than dimensions) -- this is precisely the regime "
      f"where a single neuron reflects a mix of feature directions, and where an overcomplete "
      f"SAE basis has real compressed structure worth unpacking.")

## From geometry to circuits: the induction head

Superposition explains why individual units are hard to read; the **induction circuit** (Section 16.3.1) is the canonical example of how interpretability researchers nonetheless recover a clean causal mechanism spanning *two* attention heads composed in sequence — a case of **K-composition**, where one head's output feeds directly into a second head's key computation:

- **Head 1 (previous-token head):** at position $t$, attends almost entirely to position $t-1$ and copies that token's identity forward into a dedicated slot in the residual stream.
- **Head 2 (induction head):** queries with the *current* token and uses keys built from head 1's previous-token slot — so it fires on any earlier position $s$ where token $s{-}1$ matches the current token, and copies token $s$ forward as its prediction. On a sequence like `[7, 3, 9, 5, 2, 8, 4, 7, 0]`, the bigram $(7 \to 3)$ seen at positions $0$–$1$ lets the circuit predict $3$ again the next time token $7$ recurs (position 7), by attending back to position $1$.

We will then use **activation patching** (Definitions 16.4–16.5) to test this causal story directly: *ablating* head 1's output should destroy the correct prediction (necessity), and *patching* the clean head-1 output into an otherwise-corrupted run should restore it (sufficiency).

In [ ]:
class InductionCircuit:
    """Two attention heads, hand-set weights, no training. d_model = 2V+T.
    Residual layout: [0:V]=current-token one-hot, [V:V+T]=position one-hot,
    [V+T:2V+T]=previous-token slot written by head 1.
    Head 1 (previous-token): query=position t, key=position s+1, fires at s=t-1,
      copies token(s) into the previous-token slot.
    Head 2 (induction): query=CURRENT token, key=PREVIOUS-TOKEN SLOT from head 1
      (this is K-composition: head 2's key depends on head 1's output).
      Copies token(s) forward as the prediction.
    """

    def __init__(self, vocab, seq_len, beta=30.0):
        self.V, self.T, self.beta = vocab, seq_len, beta

    def embed(self, tokens):
        V, T = self.V, self.T
        r = np.zeros((len(tokens), 2 * V + T))
        r[np.arange(len(tokens)), tokens] = 1.0
        r[np.arange(len(tokens)), V + np.arange(len(tokens))] = 1.0
        return r

    def _attend(self, Q, K, causal_strict=True):
        T = Q.shape[0]
        scores = self.beta * (Q @ K.T)
        mask = (np.arange(T)[None, :] >= np.arange(T)[:, None] if causal_strict
                else np.arange(T)[None, :] > np.arange(T)[:, None])
        scores = np.where(mask, -np.inf, scores)
        scores[0, :] = -np.inf
        finite = np.isfinite(scores)
        rowmax = np.where(finite.any(axis=1, keepdims=True),
                           np.max(np.where(finite, scores, -np.inf), axis=1, keepdims=True), 0.0)
        out = np.where(finite, np.exp(np.where(finite, scores - rowmax, 0.0)), 0.0)
        denom = out.sum(axis=1, keepdims=True)
        return np.divide(out, denom, out=np.zeros_like(out), where=denom > 0)

    def head1(self, r):
        V, T = self.V, self.T
        pos = r[:, V:V + T]
        shifted = np.zeros_like(pos); shifted[:, 1:] = pos[:, :-1]
        A = self._attend(pos, shifted)
        moved = A @ r[:, :V]
        delta = np.zeros_like(r); delta[:, V + T:] = moved
        return delta, A

    def head2(self, r):
        V, T = self.V, self.T
        Q = r[:, :V]; K = r[:, V + T:]
        A = self._attend(Q, K)
        return A @ r[:, :V], A

    def run(self, tokens, ablate_head1=False, patch_head1=None):
        r = self.embed(tokens)
        delta, A1 = self.head1(r)
        if ablate_head1:
            delta = np.zeros_like(delta)
        if patch_head1 is not None:
            delta = patch_head1
        r = r + delta
        logits, A2 = self.head2(r)
        return logits, delta, A1, A2


V, T = 12, 9
tokens = [7, 3, 9, 5, 2, 8, 4, 7, 0]
circuit = InductionCircuit(vocab=V, seq_len=T)

logits_clean, delta_clean, A1_clean, A2_clean = circuit.run(tokens)
pred_pos = 7  # second occurrence of token 7; bigram target is token 3
predicted_token = int(np.argmax(logits_clean[pred_pos]))
print(f"Sequence: {tokens}")
print(f"At position {pred_pos} (token={tokens[pred_pos]}), circuit predicts token "
      f"{predicted_token} (expected: 3, the bigram completion from positions 0-1).")
print(f"Attention mass from head2 at position {pred_pos} onto position 1: "
      f"{A2_clean[pred_pos, 1]:.4f}")

## Visualizing the attention patterns and the patching result

The two attention matrices tell the mechanistic story visually: $A_1$ should show a clean **shifted diagonal** (each position attending to the position immediately before it), while $A_2$ should show a **long-range spike** at row 7, column 1 — the induction jump. We then run the necessity/sufficiency protocol from Definitions 16.4–16.5: a **clean** run (baseline), a **corrupted** run (head 1 ablated — destroys the previous-token mechanism), and a **denoised** run (clean head-1 activations patched into the corrupted run) — reporting the logit margin for the correct token (`token 3` minus the mean of all other logits) in each case.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

im0 = axes[0].imshow(A1_clean, cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("Head 1 attention $A_1$\n(previous-token: shifted diagonal)")
axes[0].set_xlabel("key position"); axes[0].set_ylabel("query position")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(A2_clean, cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("Head 2 attention $A_2$\n(induction: long-range spike)")
axes[1].set_xlabel("key position"); axes[1].set_ylabel("query position")
axes[1].annotate("row 7 -> col 1\n(induction jump)", xy=(1, 7), xytext=(3.5, 7.6),
                  color="white", fontsize=9, arrowprops=dict(arrowstyle="->", color="white"))
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()


def logit_margin(logits, row, correct_token):
    others = np.delete(logits[row], correct_token)
    return logits[row, correct_token] - others.mean()


correct_token = 3
logits_corrupt, _, A1_corrupt, _ = circuit.run(tokens, ablate_head1=True)
# Denoising: run with head 1's *clean* output patched in directly, simulating
# "otherwise-corrupted run + clean head-1 activations restored" (Definition 16.4).
logits_denoised, _, _, _ = circuit.run(tokens, patch_head1=delta_clean)

margin_clean = logit_margin(logits_clean, pred_pos, correct_token)
margin_corrupt = logit_margin(logits_corrupt, pred_pos, correct_token)
margin_denoised = logit_margin(logits_denoised, pred_pos, correct_token)

print(f"{'run':>12} | {'pred token':>10} | {'logit margin (token 3)':>24}")
print("-" * 54)
for name, lg, m in [("clean", logits_clean, margin_clean),
                     ("corrupted", logits_corrupt, margin_corrupt),
                     ("denoised", logits_denoised, margin_denoised)]:
    pred = int(np.argmax(lg[pred_pos]))
    print(f"{name:>12} | {pred:10d} | {m:24.4f}")

print(f"\nAblating head 1 (necessity): margin drops from {margin_clean:.4f} to "
      f"{margin_corrupt:.4f} -- the previous-token mechanism is necessary for the correct prediction.")
print(f"Patching clean head-1 output back in (sufficiency): margin is restored to "
      f"{margin_denoised:.4f}, {'matching' if abs(margin_denoised - margin_clean) < 1e-6 else 'nearly matching'} "
      f"the clean run -- head 1's output alone is sufficient to recover the behavior.")

## Why induction heads are a big deal in production LLMs

Induction heads are not a quirk of this toy 2-head construction — Olsson et al. (2022) found that virtually **every transformer studied, across scales, develops induction heads during a sharp, identifiable "phase transition"** early in training, visible as a bump in the training loss curve and a simultaneous jump in in-context-learning ability. The mechanism this notebook hand-built (previous-token head feeding an induction head via K-composition) is the same circuit real models discover through gradient descent alone — it is currently the best-understood multi-component circuit in any trained transformer, and it is believed to be a major contributor to few-shot, in-context learning: a model that can complete "A B ... A" with "B" for *arbitrary* tokens A, B is doing pattern completion, which is exactly what in-context learning needs.

The activation-patching protocol used above (ablate for necessity, patch for sufficiency) is likewise not a toy technique — it is the standard causal-intervention methodology used across the interpretability literature (and inside labs like Anthropic, Google DeepMind, and EleutherAI) to validate *any* claimed circuit in a real model, from induction heads to IOI (indirect object identification) to arithmetic circuits: find a clean and corrupted input pair that differ only in the behavior of interest, then patch individual components' activations between the two runs to localize which components are causally load-bearing, rather than merely correlated with the behavior.

In [ ]:
# Numeric check: does the induction mechanism generalize to ARBITRARY bigrams,
# not just the one hand-picked sequence above? If induction heads implement
# general pattern-completion (the claimed source of in-context learning), the
# circuit should recover the correct completion for many random (A, B, recall)
# triples, not just token 7 -> 3.

rng = np.random.default_rng(0)
n_trials = 50
successes = 0
margins_random = []

for _ in range(n_trials):
    a, b = rng.choice(V, size=2, replace=False)
    filler = rng.choice([t for t in range(V) if t not in (a, b)], size=T - 3, replace=True)
    seq = [int(a), int(b)] + list(map(int, filler)) + [int(a)]  # A, B, ..., A  (recall B)
    seq = seq[:T]
    logits, _, _, _ = circuit.run(seq)
    query_pos = T - 1
    pred = int(np.argmax(logits[query_pos]))
    margins_random.append(logit_margin(logits, query_pos, b))
    successes += int(pred == b)

print(f"Induction recall accuracy over {n_trials} random (A, B, filler) sequences: "
      f"{successes}/{n_trials} = {successes / n_trials:.1%}")
print(f"Mean logit margin for the correct recalled token: {np.mean(margins_random):.4f} "
      f"(std {np.std(margins_random):.4f})")
print("\nThe circuit generalizes far beyond the single hand-picked example: it implements "
      "genuine pattern completion (A, B, ..., A -> B) for arbitrary token pairs, which is "
      "precisely the abstraction real induction heads are believed to provide inside LLMs "
      "for few-shot in-context learning.")

## Exercise

1. The toy superposition model above uses a fixed importance schedule $I_i = 0.9^i$. Rederive the phase-change curve with a *uniform* importance ($I_i = 1$ for all $i$) and with a much steeper schedule (e.g. $I_i = 0.5^i$). Does the sparsity level at which superposition "turns on" shift, and does the *identity* of which features get private axes at low sparsity change in the way Theorem 16.1's packing argument predicts? The near-orthogonal packing bound of Theorem 16.1, Eqs. (16.1)–(16.4), is the background for why it behaves this way.

2. Extend the activation-patching experiment to a **noising** intervention (Definition 16.5, the counterpart to the denoising case used above): start from the *clean* run and patch in the *corrupted* (ablated) head-1 output, then report the logit margin. Does noising damage the prediction by roughly the same amount that denoising restores it, and what would a asymmetry between the two tell you about whether head 1's output is the *only* causal pathway to the correct prediction versus one of several redundant pathways? §16.3.2 (activation patching) and §16.5 (honest limits) are directly relevant to interpreting an asymmetric result.